### Structured Output
Models can be requested to provide their response in a format matching a given schema. This is useful for ensuring the output can be easily parsed and used in subsequent processing. Langchain supports multiple schema types and methods for enforcing structured output.

### Pydantic

Pydantic models provide the richest feature set with field validation, descriptions, and nested structures.

In [6]:
import os
from langchain.chat_models import init_chat_model
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:qwen/qwen3-32b")
model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.11'}}, output_version=None, profile={'name': 'Qwen3 32B', 'release_date': '2024-12-23', 'last_updated': '2024-12-23', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 40960, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x0000025DC3008190>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000025DC3008B90>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [7]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title:str=Field(description="The title of movie")
    year:int=Field(description="The movie was released this year")
    director:str=Field(description="The director of the movie")
    rating:float=Field(description="Rating of the movie")

In [13]:
model_with_structure = model.with_structured_output(Movie)
model_with_structure

_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.11'}}, output_version=None, profile={'name': 'Qwen3 32B', 'release_date': '2024-12-23', 'last_updated': '2024-12-23', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 40960, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x0000025DC3008190>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000025DC3008B90>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title':

In [14]:
model_with_structure.invoke("Provide details of the movie Inception")

Movie(title='Inception', year=2010, director='Christopher Nolan', rating=8.8)

In [15]:
model.invoke("Provide details of the movie Inception")

AIMessage(content='<think>\nOkay, so I need to provide details about the movie Inception. Let me start by recalling what I know. Inception is a sci-fi action movie directed by Christopher Nolan, right? It\'s the one with the dream-sharing concept. The main character is Dom Cobb, played by Leonardo DiCaprio. He\'s a thief who can steal secrets by entering people\'s dreams. The movie\'s tagline is something like "Heists are dangerous. Dreams are dangerous." \n\nThe plot revolves around the idea of planting an idea into someone\'s subconscious, which is called "inception." Cobb\'s team is trying to do this for a wealthy businessman named Robert Fischer. But there\'s a catch because they have to navigate through multiple layers of dreams, each deeper than the last, and each layer slows down time exponentially. That part is a bit complicated, but I think the deeper you go, the more time passes in the real world, so they have to be careful not to get stuck in the dream too long.\n\nThe suppo

### Message output alongside parsed structure

In [16]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title:str=Field(description="The title of movie")
    year:int=Field(description="The movie was released this year")
    director:str=Field(description="The director of the movie")
    rating:float=Field(description="Rating of the movie")

model_with_structure = model.with_structured_output(Movie, include_raw=True)
response = model_with_structure.invoke("Provide details of the movie Inception")
response

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking for details about the movie Inception. Let me see what I need to do. The available tool is the Movie function, which requires title, year, director, and rating. I need to make sure I have all these parameters.\n\nFirst, the title is obviously "Inception". The year it was released... I think that\'s 2010. The director is Christopher Nolan, right? And the rating... maybe the IMDb rating, which I believe is around 8.8. Let me double-check those details to be sure. Yeah, Inception came out in 2010, directed by Christopher Nolan, and it has an 8.8 rating on IMDb. Okay, that\'s all the required info. Time to structure the tool call with those parameters.\n', 'tool_calls': [{'id': 'xymtvsary', 'function': {'arguments': '{"director":"Christopher Nolan","rating":8.8,"title":"Inception","year":2010}', 'name': 'Movie'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 213,

In [20]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name:str
    role:str

class MovieDetails(BaseModel):
    title:str=Field(description="Movie Title")
    year:int
    cast:list[Actor]
    geners:list[str]
    budget:float | None = Field(None, description="Budget in Millions USD")

model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide details about Inception")

In [21]:
response

MovieDetails(title='Inception', year=2010, cast=[Actor(name='Leonardo DiCaprio', role='Dom Cobb'), Actor(name='Joseph Gordon-Levitt', role='Arthur'), Actor(name='Ellen Page', role='Mary Arthur'), Actor(name='Tom Hardy', role='Joshua'), Actor(name='Ken Watanabe', role='Saito')], geners=['Action', 'Science Fiction', 'Thriller'], budget=160.0)

### TypedDict
TypedDict provides a simpler alternative using Python's builtin typing, ideal when you don't need runtime validation

In [26]:
from typing_extensions import TypedDict, Annotated

class MovieDict(TypedDict):
    """A movie wiht details"""
    title:Annotated[str,...,"Title of movie"]
    year:Annotated[int,...,"Movie Year"]
    director:Annotated[str,...,"Movie Director"]
    rating:Annotated[float,...,"Ratings"]
    

model_with_typedDict = model.with_structured_output(MovieDict)    
response = model_with_typedDict.invoke("Give details about Titanic")
response

{'director': 'James Cameron', 'rating': 7.8, 'title': 'Titanic', 'year': 1997}

In [32]:
from typing_extensions import TypedDict

class Actor(TypedDict):
    name:str
    role:str
    gender:str


class MovieDetails(TypedDict):
    title:str
    year:int
    cast:list[Actor]
    geners:list[str]
    budget:float | None = Field(None, description="Budget in Millions USD")

model_with_typedDict = model.with_structured_output(MovieDetails)

response = model_with_typedDict.invoke("Provide details about Titanic")
response

{'budget': 200000000,
 'cast': [{'gender': 'Male', 'name': 'Leonardo DiCaprio', 'role': 'Jack'},
  {'gender': 'Female', 'name': 'Kate Winslet', 'role': 'Rose'}],
 'geners': ['Drama', 'Romance'],
 'title': 'Titanic',
 'year': 1997}

In [31]:
model.profile

{'name': 'Qwen3 32B',
 'release_date': '2024-12-23',
 'last_updated': '2024-12-23',
 'open_weights': True,
 'max_input_tokens': 131072,
 'max_output_tokens': 40960,
 'text_inputs': True,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'text_outputs': True,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True,
 'attachment': False,
 'temperature': True}

### DataClasses

A Data class is a class typically containing mainly data, although there aren't really any restrictions. it is created using @dataclass decorator

In [1]:
import os
os.environ["OPENAI_API_KEY"]=os.getenv("OPENAI_API_KEY")

In [38]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent

class ContactInfo(BaseModel):
    """Contact Information of a person"""
    name:str = Field(description="Person's Name")
    email:str=Field(description="Person's Email")
    phone:str=Field(description="Person's Phone")

agent = create_agent(
    model="gpt-5",
    response_format=ContactInfo # Autoselects provider strategy
)   

result = agent.invoke({
    "messages": [{"role":"user","content":"Extract contact info from: John Doe, john@example.com,512-23456"}]
})

result

{'messages': [HumanMessage(content='Extract contact info from: John Doe, john@example.com,512-23456', additional_kwargs={}, response_metadata={}, id='481e8e91-ea6b-44c1-846d-44c39ef5f2c2'),
  AIMessage(content='{"name":"John Doe","email":"john@example.com","phone":"512-23456"}', additional_kwargs={'parsed': None, 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 480, 'prompt_tokens': 192, 'total_tokens': 672, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 448, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DvLI7F6VKxwfjFMVAGNyTUwkoT7as', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019f08ca-62ed-7e20-9d00-3d13d63ad731-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 192, 'output_tokens': 4

In [39]:
result["structured_response"]

ContactInfo(name='John Doe', email='john@example.com', phone='512-23456')

In [40]:
# Using TypedDict
from typing_extensions import TypedDict
from langchain.agents import create_agent

class ContactInfo(TypedDict):
    """Contact Information of a person"""
    name:str 
    email:str
    phone:str 

agent = create_agent(
    model="gpt-5",
    response_format=ContactInfo # Autoselects provider strategy
)   

result = agent.invoke({
    "messages": [{"role":"user","content":"Extract contact info from: John Doe, john@example.com,512-23456"}]
})

result['structured_response']

{'name': 'John Doe', 'email': 'john@example.com', 'phone': '512-23456'}

In [42]:
from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass
class ContactInfo:
    """Contact info of a person"""
    name:str
    email:str
    phone:str

agent = create_agent(
    model="gpt-5",
    response_format=ContactInfo
)     


result = agent.invoke({
    "messages": [{
        "role":"user", "content": "Extract info from hi ahmed how are you, i contacted you on ahmed@example.com but you didn't reply, I then called you at 532-998765"
                  }]
    })

result['structured_response']

ContactInfo(name='Ahmed', email='ahmed@example.com', phone='532-998765')